# Veryfi OCR Tool Test

This notebook tests the Veryfi OCR integration for document conversion.

In [ ]:
!poetry install

In [ ]:
# Install veryfi library using Poetry
!poetry add veryfi python-dotenv

In [1]:
import os
import json
from pathlib import Path
from veryfi import Client
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✓ Imports successful")

✓ Imports successful


In [2]:
# Veryfi API Credentials
# Get credentials from environment variables or set them here

VERYFI_CLIENT_ID = os.getenv('VERYFI_CLIENT_ID', 'your_client_id')
VERYFI_CLIENT_SECRET = os.getenv('VERYFI_CLIENT_SECRET', 'your_client_secret')
VERYFI_USERNAME = os.getenv('VERYFI_USERNAME', 'your_username')
VERYFI_API_KEY = os.getenv('VERYFI_API_KEY', 'your_api_key')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your_openai_api_key')

print("✓ Credentials loaded")
print(f"Client ID: {VERYFI_CLIENT_ID[:10]}..." if len(VERYFI_CLIENT_ID) > 10 else "Client ID not set")

✓ Credentials loaded
Client ID: vrfzf2ZEfu...


In [3]:
# Initialize Veryfi Client
veryfi_client = Client(
    client_id=VERYFI_CLIENT_ID,
    client_secret=VERYFI_CLIENT_SECRET,
    username=VERYFI_USERNAME,
    api_key=VERYFI_API_KEY
)

print("✓ Veryfi client initialized successfully")

✓ Veryfi client initialized successfully


## Test Function: OCR Document Processing

## Image Analysis - Quick Start

Process and analyze receipt/invoice images (JPG, PNG, etc.)

In [ ]:
def analyze_image(image_path: str):
    """
    Analyze a receipt or invoice image and extract key information.
    
    Args:
        image_path: Path to the image file (jpg, png, jpeg, etc.)
    
    Returns:
        dict: Extracted data from the image
    """
    try:
        print(f" Analyzing image: {os.path.basename(image_path)}")
        print("=" * 60)
        
        # Check if file exists
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")
        
        # Process the image
        response = veryfi_client.process_document(
            file_path=image_path,
            categories=['Receipt', 'Invoice', 'Financial']
        )

        print(response)
        
        print("✓ Image processed successfully!\n")
        
        # Extract and display key information
        vendor_name = response.get('vendor', {}).get('name', 'Unknown')
        date = response.get('date', 'N/A')
        total = response.get('total', 0)
        currency = response.get('currency_code', 'USD')
        category = response.get('category', 'N/A')
        
        print(f"Vendor: {vendor_name}")
        print(f"Date: {date}")
        print(f"Total: {currency} {total:.2f}")
        print(f"Category: {category}")
        
        # Line items
        line_items = response.get('line_items', [])
        if line_items:
            print(f"Items Found: {len(line_items)}")
            for idx, item in enumerate(line_items[:5], 1):
                desc = item.get('description', 'N/A')
                qty = item.get('quantity', 1)
                price = item.get('total', 0)
                print(f"  {idx}. {desc}")
                print(f"     Qty: {qty} × ${price:.2f}")
            
            if len(line_items) > 5:
                print(f"\n  ... and {len(line_items) - 5} more items")
        
        print("\n" + "=" * 60)
        
        return response
        
    except Exception as e:
        print(f" Error analyzing image: {str(e)}")
        raise

print("✓ Image analysis function ready")

## Example 3: Using Helper Functions

In [ ]:
# Analyze a receipt or invoice image
# Replace with your image path
# image_path = "/path/to/your/receipt.jpg" 

res = analyze_image(image_path="/Users/vishnum/Downloads/WhatsApp Image 2026-01-27 at 8.38.14 PM.jpeg") # or .png, .jpeg
print(res)
# Process the image
# response = analyze_image(image_path)

# Optional: Save the full response
# with open("analysis_result.json", "w") as f:
#     json.dump(response, f, indent=2)
#     print("💾 Full analysis saved to analysis_result.json")
print("💡 Update image_path with your image file and uncomment to analyze")

## Document Analysis with LangChain

Analyze PDF and other documents using LangChain document loaders and LLM

In [5]:
# Install required packages for document analysis
!poetry add langchain langchain-community langchain-openai pypdf python-docx
# !brew install llvm
!poetry add langchain-text-splitters
# !poetry add unstructured

print("✓ LangChain packages installed")

The following packages are already present in the pyproject.toml and will be skipped:

  - langchain
  - langchain-community
  - langchain-openai
  - pypdf
  - python-docx

If you want to update it to the latest compatible version, you can use `poetry update package`.
If you prefer to upgrade it to the latest available version, you can use `poetry add package@latest`.

Nothing to add.
The following packages are already present in the pyproject.toml and will be skipped:

  - langchain-text-splitters

If you want to update it to the latest compatible version, you can use `poetry update package`.
If you prefer to upgrade it to the latest available version, you can use `poetry add package@latest`.

Nothing to add.
✓ LangChain packages installed


In [6]:
# Import LangChain components
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate



print("✓ LangChain imports successful")

✓ LangChain imports successful


In [7]:
# Configure OpenAI API Key
# OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your_openai_api_key')

# Initialize LLM
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",  # or "gpt-3.5-turbo" for faster/cheaper
    temperature=0.7
)

print("✓ LLM initialized")
print(f"Model: gpt-4o-mini")

✓ LLM initialized
Model: gpt-4o-mini


In [8]:
def load_document(file_path: str):
    """
    Load document using appropriate LangChain loader based on file extension.
    
    Args:
        file_path: Path to the document file
        
    Returns:
        list: List of Document objects
    """
    try:
        print(f"📂 Loading document: {os.path.basename(file_path)}")
        print("=" * 60)
        
        # Get file extension
        _, ext = os.path.splitext(file_path.lower())
        
        # Choose appropriate loader
        if ext == '.pdf':
            print("📄 Using PDF loader...")
            loader = PyPDFLoader(file_path)
        elif ext == '.txt':
            print("📝 Using Text loader...")
            loader = TextLoader(file_path)
        elif ext in ['.docx', '.doc']:
            print("📃 Using DOCX loader...")
            loader = Docx2txtLoader(file_path)
        else:
            print(f"📋 Using Unstructured loader for {ext}...")
            loader = UnstructuredFileLoader(file_path)
        
        # Load the document
        documents = loader.load()
        
        print(f"✓ Document loaded successfully!")
        print(f"✓ Total pages/sections: {len(documents)}")
        
        # Calculate total content length
        total_chars = sum(len(doc.page_content) for doc in documents)
        print(f"✓ Total characters: {total_chars:,}")
        print(f"✓ Estimated words: {total_chars // 5:,}")
        
        return documents
        
    except Exception as e:
        print(f"❌ Error loading document: {str(e)}")
        raise


def print_document_content(documents, max_chars: int = 1000):
    """
    Print preview of document content.
    
    Args:
        documents: List of Document objects
        max_chars: Maximum characters to display per page
    """
    print("\n" + "=" * 60)
    print("DOCUMENT CONTENT PREVIEW")
    print("=" * 60)
    
    for idx, doc in enumerate(documents[:3], 1):  # Show first 3 pages
        content = doc.page_content[:max_chars]
        metadata = doc.metadata
        
        print(f"\n📄 Page/Section {idx}:")
        print(f"Metadata: {metadata}")
        print(f"\nContent:\n{content}")
        
        if len(doc.page_content) > max_chars:
            remaining = len(doc.page_content) - max_chars
            print(f"\n... ({remaining:,} more characters)")
        
        print("-" * 60)
    
    if len(documents) > 3:
        print(f"\n... and {len(documents) - 3} more pages/sections")
    
    print("=" * 60)


def chunk_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    Split documents into smaller chunks for processing.
    
    Args:
        documents: List of Document objects
        chunk_size: Size of each chunk in characters
        chunk_overlap: Overlap between chunks
        
    Returns:
        list: List of chunked documents
    """
    try:
        print(f"\n🔪 Chunking documents...")
        print(f"Chunk size: {chunk_size} characters")
        print(f"Chunk overlap: {chunk_overlap} characters")
        print("-" * 60)
        
        # Initialize text splitter
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        # Split documents
        chunks = text_splitter.split_documents(documents)
        
        print(f"✓ Created {len(chunks)} chunks")
        
        # Show chunk statistics
        chunk_lengths = [len(chunk.page_content) for chunk in chunks]
        print(f"✓ Average chunk size: {sum(chunk_lengths) // len(chunks)} characters")
        print(f"✓ Min chunk size: {min(chunk_lengths)} characters")
        print(f"✓ Max chunk size: {max(chunk_lengths)} characters")
        
        return chunks
        
    except Exception as e:
        print(f"❌ Error chunking documents: {str(e)}")
        raise

print("✓ Document loading and chunking functions ready")

✓ Document loading and chunking functions ready


In [ ]:
def analyze_document_with_llm(chunks, analysis_type: str = "summary"):
    """
    Analyze document chunks using LLM and provide insights.
    
    Args:
        chunks: List of document chunks
        analysis_type: Type of analysis - "summary", "insights", "key_points", "financial"
        
    Returns:
        dict: Analysis results
    """
    try:
        print(f"\n🤖 Analyzing document with LLM...")
        print(f"Analysis type: {analysis_type}")
        print("=" * 60)
        
        # Combine chunks into full text (or analyze separately for very long docs)
        full_text = "\n\n".join([chunk.page_content for chunk in chunks[:10]])  # Limit to first 10 chunks
        
        # Define prompts based on analysis type
        prompts = {
            "summary": """
            Please provide a comprehensive summary of the following document:
            
            Document:
            {text}
            
            Summary (include main topics, key information, and overall purpose):
            """,
            
            "insights": """
            Analyze the following document and provide key insights:
            
            Document:
            {text}
            
            Please provide:
            1. Main Insights (3-5 key takeaways)
            2. Important Details
            3. Actionable Items (if any)
            4. Recommendations
            """,
            
            "key_points": """
            Extract the key points from the following document:
            
            Document:
            {text}
            
            Key Points (list the most important information):
            """,
            
            "financial": """
            Analyze this financial document and provide insights:
            
            Document:
            {text}
            
            Please provide:
            1. Document Type
            2. Key Financial Figures
            3. Important Dates
            4. Summary of Financial Information
            5. Any Red Flags or Important Notes
            """
        }
        
        # Get appropriate prompt
        prompt_template = prompts.get(analysis_type, prompts["summary"])
        prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
        
        # Create chain using LCEL (LangChain Expression Language)
        chain = prompt | llm
        
        # Run analysis
        print(f"Processing {len(chunks[:10])} chunks...")
        response = chain.invoke({"text": full_text})
        result = response.content
        
        print("✓ Analysis complete!")
        print("=" * 60)
        
        return {
            "analysis_type": analysis_type,
            "result": result,
            "chunks_analyzed": len(chunks[:10]),
            "total_chunks": len(chunks)
        }
        
    except Exception as e:
        print(f"❌ Error analyzing document: {str(e)}")
        raise


def print_analysis_results(analysis):
    """Print formatted analysis results."""
    print("\n" + "=" * 60)
    print(f"📊 DOCUMENT ANALYSIS RESULTS - {analysis['analysis_type'].upper()}")
    print("=" * 60)
    
    print(f"\n{analysis['result']}")
    
    print("\n" + "-" * 60)
    print(f"Chunks analyzed: {analysis['chunks_analyzed']} of {analysis['total_chunks']}")
    print("=" * 60)


def get_document_insights(file_path: str, analysis_types: list = None):
    """
    Complete workflow: Load document, chunk it, and analyze with LLM.
    
    Args:
        file_path: Path to document file
        analysis_types: List of analysis types to perform
        
    Returns:
        dict: Complete analysis results
    """
    if analysis_types is None:
        analysis_types = ["summary", "insights"]
    
    try:
        print("🚀 Starting document analysis workflow...")
        print("=" * 60)
        
        # Step 1: Load document
        print("\n📥 STEP 1: Loading Document")
        documents = load_document(file_path)
        
        # Step 2: Print content preview
        print("\n📖 STEP 2: Document Content Preview")
        print_document_content(documents, max_chars=500)
        
        # Step 3: Chunk documents
        print("\n✂️ STEP 3: Chunking Document")
        chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=200)
        
        # Step 4: Analyze with LLM
        print("\n🔍 STEP 4: LLM Analysis")
        results = {}
        
        for analysis_type in analysis_types:
            print(f"\n--- Performing {analysis_type} analysis ---")
            analysis = analyze_document_with_llm(chunks, analysis_type)
            results[analysis_type] = analysis
            print_analysis_results(analysis)
        
        print("\n" + "=" * 60)
        print("✅ DOCUMENT ANALYSIS COMPLETE!")
        print("=" * 60)
        
        return {
            "file_path": file_path,
            "num_pages": len(documents),
            "num_chunks": len(chunks),
            "analyses": results
        }
        
    except Exception as e:
        print(f"❌ Workflow error: {str(e)}")
        raise

print("✓ LLM analysis functions ready")

✓ LLM analysis functions ready


### Example 1: Quick Document Analysis

Analyze any document (PDF, DOCX, TXT) with one function call

In [10]:
# Quick analysis - provide document path and get insights
document_path = "/Users/vishnum/Downloads/Pradeep Yellanki.pdf"  # or .docx, .txt

# Uncomment to run:
results = get_document_insights(
    file_path=document_path,
    analysis_types=["summary", "insights"]
)

# Save results
# with open("document_analysis.json", "w") as f:
#     json.dump(results, f, indent=2)
#     print("\n💾 Results saved to document_analysis.json")

print("💡 Update document_path and uncomment to analyze your document")

🚀 Starting document analysis workflow...

📥 STEP 1: Loading Document
📂 Loading document: Pradeep Yellanki.pdf
📄 Using PDF loader...
✓ Document loaded successfully!
✓ Total pages/sections: 92
✓ Total characters: 82,638
✓ Estimated words: 16,527

📖 STEP 2: Document Content Preview

DOCUMENT CONTENT PREVIEW

📄 Page/Section 1:
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20251128114420', 'source': '/Users/vishnum/Downloads/Pradeep Yellanki.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1'}

Content:
Enquiry Control Number (ECN): 9595090376
Report Date: 12 Sep 2025
Back to Top Credit Factors
Hey Pradeep Yellanki,
Here is your Credit Health Report
Powered by
762
Report Date : 12 Sep 2025
Well Done!
You have done well, but your credit health can be better.  
You may not be eligible for the best loan and credit card o ers currently.  
Check your Credit Health Report & learn to build an excellent score.
Enquiry Control Number (ECN)*: 9595090376
Enquiry Control Nu

### Example 2: Step-by-Step Document Analysis

Manual control over each step of the analysis process

In [12]:
# Step-by-step analysis with full control
document_path = "/Users/vishnum/Downloads/Pradeep Yellanki.pdf"

# Uncomment to run:

# Step 1: Load document
print("Step 1: Loading document...")
documents = load_document(document_path)

# Step 2: Preview content
print("\nStep 2: Previewing content...")
print_document_content(documents, max_chars=1000)

# Step 3: Chunk documents
print("\nStep 3: Creating chunks...")
chunks = chunk_documents(
    documents,
    chunk_size=1500,  # Customize chunk size
    chunk_overlap=300   # Customize overlap
)

# Step 4: Analyze with different types
print("\nStep 4: Running analysis...")

# Get summary
summary_analysis = analyze_document_with_llm(chunks, analysis_type="summary")
print_analysis_results(summary_analysis)

# # Get key points
# keypoints_analysis = analyze_document_with_llm(chunks, analysis_type="key_points")
# print_analysis_results(keypoints_analysis)

# # For financial documents
# financial_analysis = analyze_document_with_llm(chunks, analysis_type="financial")
# print_analysis_results(financial_analysis)

print("\n✅ Complete!")


print("💡 Update document_path and uncomment to run step-by-step analysis")

Step 1: Loading document...
📂 Loading document: Pradeep Yellanki.pdf
📄 Using PDF loader...
✓ Document loaded successfully!
✓ Total pages/sections: 92
✓ Total characters: 82,638
✓ Estimated words: 16,527

Step 2: Previewing content...

DOCUMENT CONTENT PREVIEW

📄 Page/Section 1:
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20251128114420', 'source': '/Users/vishnum/Downloads/Pradeep Yellanki.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1'}

Content:
Enquiry Control Number (ECN): 9595090376
Report Date: 12 Sep 2025
Back to Top Credit Factors
Hey Pradeep Yellanki,
Here is your Credit Health Report
Powered by
762
Report Date : 12 Sep 2025
Well Done!
You have done well, but your credit health can be better.  
You may not be eligible for the best loan and credit card o ers currently.  
Check your Credit Health Report & learn to build an excellent score.
Enquiry Control Number (ECN)*: 9595090376
Enquiry Control Number (ECN), is a Unique Report Identi cation N

### Example 5: Complete Workflow Test

Test the entire document analysis pipeline

In [ ]:
# Complete workflow test
test_document = "/path/to/your/document.pdf"

# Uncomment to run complete workflow:
"""
print("="*80)
print("COMPLETE DOCUMENT ANALYSIS WORKFLOW")
print("="*80)

# Analyze document with all analysis types
results = get_document_insights(
    file_path=test_document,
    analysis_types=["summary", "insights", "key_points", "financial"]
)

# Display results
print("\n\n")
print("="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print(f"\n📄 Document: {os.path.basename(test_document)}")
print(f"📊 Pages: {results['num_pages']}")
print(f"🔪 Chunks: {results['num_chunks']}")

print("\n📋 Available Analyses:")
for analysis_type in results['analyses'].keys():
    print(f"   ✓ {analysis_type.title()}")

# Save complete results
output_file = "complete_document_analysis.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n💾 Complete results saved to: {output_file}")

# Create a formatted report
report_file = "document_analysis_report.txt"
with open(report_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("DOCUMENT ANALYSIS REPORT\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"Document: {os.path.basename(test_document)}\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Pages: {results['num_pages']}\n")
    f.write(f"Chunks: {results['num_chunks']}\n\n")
    
    for analysis_type, analysis in results['analyses'].items():
        f.write("-"*80 + "\n")
        f.write(f"{analysis_type.upper()} ANALYSIS\n")
        f.write("-"*80 + "\n\n")
        f.write(analysis['result'] + "\n\n")

print(f"📄 Formatted report saved to: {report_file}")

print("\n" + "="*80)
print("✅ WORKFLOW COMPLETE!")
print("="*80)
"""

print("💡 Update test_document path and uncomment to run complete workflow")

## Usage Tips & Best Practices

### Supported Document Types
- **PDF** - PyPDFLoader (best for standard PDFs)
- **DOCX/DOC** - Docx2txtLoader (Word documents)
- **TXT** - TextLoader (plain text files)
- **Other formats** - UnstructuredFileLoader (HTML, CSV, etc.)

### Analysis Types
1. **summary** - Comprehensive overview of the document
2. **insights** - Key takeaways and actionable items
3. **key_points** - Bullet-point list of important information
4. **financial** - Specialized analysis for financial documents (invoices, statements, etc.)

### Configuration
- **Chunk Size**: 1000-2000 characters (balance between context and processing)
- **Chunk Overlap**: 200-300 characters (maintain context between chunks)
- **LLM Model**: 
  - `gpt-4o-mini` - Fast, cost-effective
  - `gpt-4o` - More accurate, better for complex documents
  - `gpt-3.5-turbo` - Fastest, cheapest option

### Best Practices
1. **Large Documents**: For documents >20 pages, consider processing in batches
2. **Financial Documents**: Use "financial" analysis type for invoices, statements
3. **Multilingual**: LangChain loaders support multiple languages
4. **API Keys**: Store in `.env` file, never hardcode
5. **Error Handling**: Always wrap in try-except for production use

### Environment Variables
Add to your `.env` file:
```
OPENAI_API_KEY=your_openai_api_key_here
VERYFI_CLIENT_ID=your_veryfi_client_id
VERYFI_CLIENT_SECRET=your_veryfi_client_secret
VERYFI_USERNAME=your_veryfi_username
VERYFI_API_KEY=your_veryfi_api_key
```